# U1-03 · Un recomendador de películas con MovieLens

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 1, apartado 1.8*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con el botón ▶). El entrenamiento tarda alrededor de un minuto.

In [ ]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
if not os.path.exists("ml-100k.zip"):
    urllib.request.urlretrieve(url, "ml-100k.zip")
zipfile.ZipFile("ml-100k.zip").extractall()
notas = pd.read_csv("ml-100k/u.data", sep="\t",
                    names=["usuario", "peli", "nota", "fecha"])
titulos = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                      header=None, usecols=[0, 1], index_col=0)[1]
print(len(notas), "notas de", notas.usuario.nunique(), "usuarios a",
      notas.peli.nunique(), "películas")

rng = np.random.default_rng(0)
prueba = rng.random(len(notas)) < 0.2       # 20 % para evaluar
ent, pru = notas[~prueba], notas[prueba]
u, p, r = (ent[c].to_numpy() - (c != "nota")
           for c in ("usuario", "peli", "nota"))     # índices desde 0
up, pp, rp = (pru[c].to_numpy() - (c != "nota")
              for c in ("usuario", "peli", "nota"))
rmse = lambda pred: np.sqrt(np.mean((pred - rp) ** 2))
media = r.mean()
base = rmse(np.full(len(rp), media))
print(f"predecir siempre la media: RMSE {base:.3f}")

# factorización de matrices: nota ≈ media + sesgos + p_u · q_i
def entrenar(k, epocas=30, eta=0.01, lam=0.05):
    P = rng.normal(0, 0.1, (943, k))
    Q = rng.normal(0, 0.1, (1682, k))
    bu, bi = np.zeros(943), np.zeros(1682)
    for epoca in range(epocas):
        for j in rng.permutation(len(r)):        # SGD, nota a nota
            a, b = u[j], p[j]
            e = r[j] - (media + bu[a] + bi[b] + P[a] @ Q[b])
            bu[a] += eta * (e - lam * bu[a])
            bi[b] += eta * (e - lam * bi[b])
            P[a], Q[b] = (P[a] + eta * (e * Q[b] - lam * P[a]),
                          Q[b] + eta * (e * P[a] - lam * Q[b]))
    return lambda us, ps: (media + bu[us] + bi[ps]
                           + np.sum(P[us] * Q[ps], axis=1))

for k in (0, 20):            # k = 0: solo sesgos
    modelo = entrenar(k)
    print(f"k = {k:2d}: RMSE {rmse(modelo(up, pp)):.3f}")

# recomendaciones para el usuario 1 (índice 0), sin repetir vistas
vistas = set(p[u == 0])
nuevas = np.array([i for i in range(1682) if i not in vistas])
nota_est = modelo(np.zeros(len(nuevas), int), nuevas)
for i in nuevas[np.argsort(nota_est)[::-1][:5]]:
    print(f"  {titulos[i + 1]}")

**Prueba.** Cambia `k` a 50 y `epocas` a 60. ¿Mejora el error en prueba o empieza a empeorar?

**Ejercicio.** Muestra las cinco recomendaciones para otro usuario y compáralas con las películas que mejor valoró.

> Datos: MovieLens 100K, GroupLens Research, Universidad de Minnesota (Harper y Konstan, 2015, https://doi.org/10.1145/2827872). Se descargan de su web; no se pueden redistribuir.
>
> Koren, Y., Bell, R. y Volinsky, C. (2009). Matrix factorization techniques for recommender systems. *Computer*, 42(8), 30–37. https://doi.org/10.1109/MC.2009.263